In [1]:
# Imports and config

import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from config import load_settings

settings = load_settings()
folder = settings['paths']['folder']
input_path = os.path.join(folder, settings['paths']['input_file'])

AM_PEAK_PERIODS = settings['trip_filter']['am_peak_periods']
CAR_MODES = set(settings['trip_filter']['car_modes'])


In [2]:
# Load and filter data (slow — only re-run if input file changes)
# tripData.csv is ~6 GB / 48 columns; read only the columns we need, in chunks,
# filtering as we go so peak memory stays small.

USECOLS = ['depart_period', 'trip_mode_name', 'tour_purpose', 'dest_purpose']

chunks = []
for chunk in pd.read_csv(input_path, usecols=USECOLS, chunksize=2_000_000):
    chunks.append(chunk[
        chunk['depart_period'].isin(AM_PEAK_PERIODS) &
        chunk['trip_mode_name'].isin(CAR_MODES)
    ])
df_filtered = pd.concat(chunks, ignore_index=True)
del chunks

print(f"Total AM peak car trips: {len(df_filtered):,}")

Total AM peak car trips: 3,606,113


In [3]:
# Count work vs other auto trips

work_tour_mask = df_filtered['tour_purpose'].str.lower().str.startswith(('work_', 'atwork_'))
n_work  = work_tour_mask.sum()
n_other = (~work_tour_mask).sum()
total   = len(df_filtered)

print(f"Total AM peak car trips : {total:>12,}")
print(f"  Work auto trips       : {n_work:>12,}  ({100*n_work/total:.1f}%)")
print(f"  Other auto trips      : {n_other:>12,}  ({100*n_other/total:.1f}%)")
print()
print("Work trips by tour_purpose:")
print(df_filtered.loc[work_tour_mask, 'tour_purpose'].value_counts())

Total AM peak car trips :    3,606,113
  Work auto trips       :    1,393,904  (38.7%)
  Other auto trips      :    2,212,209  (61.3%)

Work trips by tour_purpose:
tour_purpose
work_whitecollar      719752
work_retailandfood    218297
work_bluecollar       202300
work_services         119989
work_health            82984
atwork_business        40221
atwork_eat              5847
atwork_maint            4514
Name: count, dtype: int64


In [4]:
# Count work vs other auto trips by dest_purpose

total = len(df_filtered)
dest_purpose_counts = df_filtered['dest_purpose'].value_counts()

n_work_dest  = dest_purpose_counts.get('work', 0) + dest_purpose_counts.get('Work', 0)
n_other_dest = total - n_work_dest

print(f"Total AM peak car trips : {total:>12,}")
print(f"  Work dest trips       : {n_work_dest:>12,}  ({100*n_work_dest/total:.1f}%)")
print(f"  Other dest trips      : {n_other_dest:>12,}  ({100*n_other_dest/total:.1f}%)")

Total AM peak car trips :    3,606,113
  Work dest trips       :    1,064,495  (29.5%)
  Other dest trips      :    2,541,618  (70.5%)


In [5]:
# Count work vs other auto trips in T171 (PECAS) trips
# TripCalculationsI.csv only holds the coefficients that define each TripType;
# the resulting trips (one row per trip, tagged by type) are in tazTrips.csv.

trip_calcs = pd.read_csv(settings['paths']['t171_trip_calculations_i'])
print("TripCalculationsI coefficients by TripType:")
print(trip_calcs.groupby('TripType')['CommodityName'].apply(list).to_string())
print()

t171_trips = pd.read_csv(settings['paths']['t171_2020_taz_trips_csv'], usecols=['type'])
t171_type_counts = t171_trips['type'].value_counts()

t171_total   = len(t171_trips)
t171_n_work  = t171_type_counts.get('work', 0)
t171_n_other = t171_total - t171_n_work

print(f"Total T171 2020 trips   : {t171_total:>12,}")
print(f"  Work trips            : {t171_n_work:>12,}  ({100*t171_n_work/t171_total:.1f}%)")
print(f"  Other trips           : {t171_n_other:>12,}  ({100*t171_n_other/t171_total:.1f}%)")

TripCalculationsI coefficients by TripType:
TripType
non-work    [CS42Whlsale, CS4445Retail, CS4849Trans, CS51I...
work        [CL01BlueCollar, CL02Health, CL03RetailandFood...

Total T171 2020 trips   :    2,633,306
  Work trips            :    1,064,394  (40.4%)
  Other trips           :    1,568,912  (59.6%)


In [6]:
# Count total trips in the OMX matrix

import openmatrix as omx
import numpy as np

omx_path = os.path.join(
    folder,
    settings['aequilibrae_network']['project_path'],
    'matrices',
    os.path.splitext(settings['paths']['trip_list_file'])[0] + '.omx'
)

with omx.open_file(omx_path, 'r') as m:
    total_omx = np.array(m['trips']).sum()

print(f"Total trips in OMX matrix: {total_omx:,.1f}")

Total trips in OMX matrix: 1,202,037.7


In [7]:
# Cell 6: Tabulate counts by dest_purpose

print(df_filtered['dest_purpose'].value_counts())

dest_purpose
work          1057013
escort         489089
school         478454
Home           439102
othmaint       420280
shopping       281480
othdiscr       199269
eatout          80998
social          78250
university      42057
atwork          32639
Work             7482
Name: count, dtype: int64


In [8]:
# Inspect trips_joint_AM.OMX

import openmatrix as omx
import numpy as np

joint_omx_path = os.path.join(folder, 'trips_joint_AM.OMX')

with omx.open_file(joint_omx_path, 'r') as m:
    print(f"Matrix names : {m.list_matrices()}")
    print(f"Shape        : {m.shape()}")
    for name in m.list_matrices():
        data = np.array(m[name])
        print(f"\n  [{name}]")
        print(f"    sum      : {data.sum():,.4f}")
        print(f"    min      : {data.min()}")
        print(f"    max      : {data.max()}")
        print(f"    nonzero  : {np.count_nonzero(data):,}")

Matrix names : ['BIKE', 'DA', 'DATOLL', 'KNRALLWLK', 'KNRPRMWLK', 'PNRALLWLK', 'PNRPRMWLK', 'SCHLBUS', 'SR2', 'SR2TOLL', 'SR3', 'SR3TOLL', 'WALK', 'WLKALLKNR', 'WLKALLPNR', 'WLKALLWLK', 'WLKPRMKNR', 'WLKPRMPNR', 'WLKPRMWLK']
Shape        : (np.int32(5922), np.int32(5922))

  [BIKE]
    sum      : 647.0000
    min      : 0.0
    max      : 8.0
    nonzero  : 238

  [DA]
    sum      : 0.0000
    min      : 0.0
    max      : 0.0
    nonzero  : 0

  [DATOLL]
    sum      : 0.0000
    min      : 0.0
    max      : 0.0
    nonzero  : 0

  [KNRALLWLK]
    sum      : 8.0000
    min      : 0.0
    max      : 2.0
    nonzero  : 4

  [KNRPRMWLK]
    sum      : 0.0000
    min      : 0.0
    max      : 0.0
    nonzero  : 0

  [PNRALLWLK]
    sum      : 5.0000
    min      : 0.0
    max      : 3.0
    nonzero  : 2

  [PNRPRMWLK]
    sum      : 30.0000
    min      : 0.0
    max      : 3.0
    nonzero  : 14

  [SCHLBUS]
    sum      : 0.0000
    min      : 0.0
    max      : 0.0
    nonzero  : 0

 

In [9]:

# Table of trips by matrix in trips_joint_AM.OMX

with omx.open_file(joint_omx_path, 'r') as m:
    rows = [{'matrix': name, 'trips': np.array(m[name]).sum()} for name in m.list_matrices()]

summary = pd.DataFrame(rows).set_index('matrix')
summary['trips'] = summary['trips'].map('{:,.1f}'.format)
print(summary.to_string())

              trips
matrix             
BIKE          647.0
DA              0.0
DATOLL          0.0
KNRALLWLK       8.0
KNRPRMWLK       0.0
PNRALLWLK       5.0
PNRPRMWLK      30.0
SCHLBUS         0.0
SR2        92,904.0
SR2TOLL        79.0
SR3        47,304.0
SR3TOLL        58.0
WALK       16,896.0
WLKALLKNR       0.0
WLKALLPNR       0.0
WLKALLWLK     804.0
WLKPRMKNR       0.0
WLKPRMPNR       0.0
WLKPRMWLK     178.0
